# Endpoints de triaje hospitalario — versión corregida

Este notebook reemplaza la versión anterior y genera un `main.py` alineado con la nueva base de datos de 15 tablas.

Incluye autenticación JWT, permisos por rol, autoría clínica, edición restringida al médico creador, antecedentes, diagnósticos, notas clínicas, exámenes, historia clínica unificada, prescripciones y facturación relacionada con paciente + encuentro.

**La capa FHIR no se mezcla todavía en este notebook.** Se añadirá después como bloque de interoperabilidad separado para que primero validemos correctamente la API de negocio.


In [1]:
%pip install -q fastapi uvicorn psycopg2-binary python-dotenv pyjwt "pwdlib[argon2]"


Note: you may need to restart the kernel to use updated packages.


## Generar `main.py`

La siguiente celda sobrescribe `main.py` con la versión corregida. Antes de ejecutarla, asegúrate de que `pass.env` tenga `PG_CONNECTION_STRING` y `JWT_SECRET_KEY`.


In [2]:
from pathlib import Path

MAIN_CODE = 'import os\nfrom datetime import datetime, timedelta, timezone\nfrom decimal import Decimal\nfrom typing import Literal\n\nimport jwt\nimport psycopg2\nfrom dotenv import load_dotenv\nfrom fastapi import Depends, FastAPI, HTTPException, Query, status\nfrom fastapi.security import HTTPAuthorizationCredentials, HTTPBearer\nfrom psycopg2.errors import UniqueViolation\nfrom psycopg2.extras import Json, RealDictCursor\nfrom pwdlib import PasswordHash\nfrom pydantic import BaseModel, Field\n\nload_dotenv("pass.env", override=True)\n\nPG_CONNECTION_STRING=os.getenv("PG_CONNECTION_STRING")\nJWT_SECRET_KEY=os.getenv("JWT_SECRET_KEY")\nJWT_ALGORITHM="HS256"\nJWT_EXPIRE_MINUTES=60\n\nif not PG_CONNECTION_STRING:\n    raise RuntimeError("Falta PG_CONNECTION_STRING en pass.env")\nif not JWT_SECRET_KEY:\n    raise RuntimeError("Falta JWT_SECRET_KEY en pass.env")\n\napp=FastAPI(\n    title="API de Triaje Hospitalario",\n    version="2.0.0",\n    description="API clínica y administrativa alineada con la BD corregida."\n)\n\nsecurity=HTTPBearer()\npassword_hash=PasswordHash.recommended()\n\ndef get_db():\n    conn=psycopg2.connect(PG_CONNECTION_STRING)\n    try:\n        yield conn\n    finally:\n        conn.close()\n\ndef jsonable_row(row):\n    if row is None:\n        return None\n    d=dict(row)\n    for k,v in d.items():\n        if isinstance(v,Decimal):\n            d[k]=float(v)\n        elif isinstance(v,datetime):\n            d[k]=v.isoformat()\n    return d\n\ndef registrar_auditoria(cursor,tabla,registro_id,accion,usuario,anteriores=None,nuevos=None):\n    cursor.execute(\n        """\n        INSERT INTO auditoria_cambios(\n            tabla_afectada,registro_id,accion,datos_anteriores,datos_nuevos,realizado_por\n        ) VALUES(%s,%s,%s,%s,%s,%s);\n        """,\n        (\n            tabla,str(registro_id),accion,\n            Json(jsonable_row(anteriores)) if anteriores else None,\n            Json(jsonable_row(nuevos)) if nuevos else None,\n            usuario\n        )\n    )\n\ndef crear_token(usuario):\n    now=datetime.now(timezone.utc)\n    payload={\n        "sub":str(usuario["numero_documento_usuario"]),\n        "username":usuario["username"],\n        "rol":usuario["rol"],\n        "iat":now,\n        "exp":now+timedelta(minutes=JWT_EXPIRE_MINUTES)\n    }\n    return jwt.encode(payload,JWT_SECRET_KEY,algorithm=JWT_ALGORITHM)\n\ndef usuario_actual(\n    credenciales:HTTPAuthorizationCredentials=Depends(security),\n    db=Depends(get_db)\n):\n    try:\n        payload=jwt.decode(credenciales.credentials,JWT_SECRET_KEY,algorithms=[JWT_ALGORITHM])\n        documento=int(payload["sub"])\n    except Exception:\n        raise HTTPException(status_code=status.HTTP_401_UNAUTHORIZED,detail="Token inválido o vencido")\n\n    cur=db.cursor(cursor_factory=RealDictCursor)\n    try:\n        cur.execute("""\n            SELECT u.numero_documento_usuario,u.username,u.id_rol,r.nombre AS rol\n            FROM usuarios u\n            JOIN roles r ON r.id_rol=u.id_rol\n            WHERE u.numero_documento_usuario=%s\n              AND u.estado=TRUE\n              AND u.is_deleted=FALSE;\n        """,(documento,))\n        u=cur.fetchone()\n    finally:\n        cur.close()\n\n    if not u:\n        raise HTTPException(status_code=401,detail="Usuario no disponible")\n\n    return dict(u)\n\ndef requerir_roles(*roles):\n    def dep(u=Depends(usuario_actual)):\n        if u["rol"] not in roles:\n            raise HTTPException(status_code=403,detail="No tiene permisos para esta operación")\n        return u\n    return dep\n\ndef es_paciente_propio(cursor,id_paciente,u):\n    cursor.execute("""\n        SELECT 1\n        FROM pacientes\n        WHERE numero_documento_paciente=%s\n          AND id_usuario=%s\n          AND is_deleted=FALSE;\n    """,(id_paciente,u["numero_documento_usuario"]))\n    return cursor.fetchone() is not None\n\ndef exigir_paciente_propio(cursor,id_paciente,u):\n    if u["rol"]=="Paciente" and not es_paciente_propio(cursor,id_paciente,u):\n        raise HTTPException(status_code=403,detail="Solo puede consultar su propia información")\n\ndef obtener_encuentro(cursor,id_encuentro,incluir_eliminado=True):\n    q="SELECT * FROM encuentros WHERE id_encuentro=%s"\n    if not incluir_eliminado:\n        q+=" AND is_deleted=FALSE"\n    cursor.execute(q+";",(id_encuentro,))\n    r=cursor.fetchone()\n    return dict(r) if r else None\n\ndef exigir_acceso_encuentro(cursor,encuentro,u):\n    if not encuentro:\n        raise HTTPException(status_code=404,detail="Encuentro no encontrado")\n    exigir_paciente_propio(cursor,encuentro["id_paciente"],u)\n\ndef exigir_autor_o_admin(registro,campo_autor,u):\n    if u["rol"]=="Admin":\n        return\n    if u["rol"]!="Medico" or registro[campo_autor]!=u["numero_documento_usuario"]:\n        raise HTTPException(\n            status_code=403,\n            detail="El médico solo puede modificar o eliminar registros creados por él mismo"\n        )\n\ndef obtener_registro(cursor,tabla,pk,id_registro):\n    cursor.execute(f"SELECT * FROM {tabla} WHERE {pk}=%s;",(id_registro,))\n    r=cursor.fetchone()\n    return dict(r) if r else None\n\n@app.get("/",tags=["Sistema"])\ndef raiz():\n    return {"mensaje":"API de Triaje Hospitalario","version":"2.0.0"}\n\n@app.get("/estado-bd",tags=["Sistema"])\ndef estado_bd(db=Depends(get_db)):\n    cur=db.cursor()\n    try:\n        cur.execute("SELECT current_database(),current_user;")\n        bd,usuario=cur.fetchone()\n        return {"estado":"ok","base_datos":bd,"usuario":usuario}\n    finally:\n        cur.close()\n\nclass LoginIn(BaseModel):\n    username:str\n    password:str\n\n@app.post("/auth/login",tags=["Autenticación"])\ndef login(datos:LoginIn,db=Depends(get_db)):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n    try:\n        cur.execute("""\n            SELECT\n                u.numero_documento_usuario,u.username,u.password_hash,\n                u.estado,u.is_deleted,r.nombre AS rol\n            FROM usuarios u\n            JOIN roles r ON r.id_rol=u.id_rol\n            WHERE u.username=%s;\n        """,(datos.username,))\n        u=cur.fetchone()\n    finally:\n        cur.close()\n\n    if (\n        not u\n        or not u["estado"]\n        or u["is_deleted"]\n        or not password_hash.verify(datos.password,u["password_hash"])\n    ):\n        raise HTTPException(status_code=401,detail="Credenciales incorrectas")\n\n    return {"access_token":crear_token(u),"token_type":"bearer","rol":u["rol"]}\n\n@app.get("/auth/me",tags=["Autenticación"])\ndef me(u=Depends(usuario_actual)):\n    return u\n\nclass UsuarioCreate(BaseModel):\n    numero_documento_usuario:int\n    id_rol:int\n    username:str=Field(min_length=3,max_length=50)\n    password:str=Field(min_length=8)\n    nombres:str=Field(min_length=1,max_length=100)\n    apellidos:str=Field(min_length=1,max_length=100)\n    email:str|None=None\n    telefono:str|None=Field(default=None,max_length=30)\n\nclass UsuarioUpdate(BaseModel):\n    username:str|None=Field(default=None,min_length=3,max_length=50)\n    nombres:str|None=Field(default=None,max_length=100)\n    apellidos:str|None=Field(default=None,max_length=100)\n    email:str|None=None\n    telefono:str|None=Field(default=None,max_length=30)\n    estado:bool|None=None\n    id_rol:int|None=None\n\n@app.post("/usuarios",tags=["Usuarios"],status_code=201)\ndef crear_usuario(data:UsuarioCreate,db=Depends(get_db),u=Depends(requerir_roles("Admin"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n    try:\n        cur.execute("SELECT 1 FROM roles WHERE id_rol=%s AND is_active=TRUE;",(data.id_rol,))\n        if not cur.fetchone():\n            raise HTTPException(status_code=400,detail="Rol inválido")\n\n        cur.execute("""\n            INSERT INTO usuarios(\n                numero_documento_usuario,id_rol,username,password_hash,\n                nombres,apellidos,email,telefono\n            )\n            VALUES(%s,%s,%s,%s,%s,%s,%s,%s)\n            RETURNING *;\n        """,(\n            data.numero_documento_usuario,data.id_rol,data.username,\n            password_hash.hash(data.password),data.nombres,data.apellidos,\n            data.email,data.telefono\n        ))\n\n        nuevo=cur.fetchone()\n        registrar_auditoria(\n            cur,"usuarios",data.numero_documento_usuario,"CREAR",\n            u["numero_documento_usuario"],nuevos=nuevo\n        )\n        db.commit()\n\n        d=dict(nuevo)\n        d.pop("password_hash",None)\n        return d\n\n    except UniqueViolation:\n        db.rollback()\n        raise HTTPException(status_code=409,detail="Documento, usuario o correo ya registrado")\n    except:\n        db.rollback()\n        raise\n    finally:\n        cur.close()\n\n@app.get("/usuarios",tags=["Usuarios"])\ndef listar_usuarios(db=Depends(get_db),u=Depends(requerir_roles("Admin"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n    try:\n        cur.execute("""\n            SELECT\n                u.numero_documento_usuario,u.username,u.nombres,u.apellidos,\n                u.email,u.telefono,u.estado,u.is_deleted,r.nombre AS rol\n            FROM usuarios u\n            JOIN roles r ON r.id_rol=u.id_rol\n            ORDER BY u.apellidos,u.nombres;\n        """)\n        return cur.fetchall()\n    finally:\n        cur.close()\n\n@app.get("/usuarios/{documento}",tags=["Usuarios"])\ndef ver_usuario(documento:int,db=Depends(get_db),u=Depends(requerir_roles("Admin"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n    try:\n        cur.execute("""\n            SELECT\n                u.numero_documento_usuario,u.username,u.nombres,u.apellidos,\n                u.email,u.telefono,u.estado,u.is_deleted,r.nombre AS rol\n            FROM usuarios u\n            JOIN roles r ON r.id_rol=u.id_rol\n            WHERE numero_documento_usuario=%s;\n        """,(documento,))\n        r=cur.fetchone()\n        if not r:\n            raise HTTPException(status_code=404,detail="Usuario no encontrado")\n        return r\n    finally:\n        cur.close()\n\n@app.put("/usuarios/{documento}",tags=["Usuarios"])\ndef editar_usuario(documento:int,data:UsuarioUpdate,db=Depends(get_db),u=Depends(requerir_roles("Admin"))):\n    cambios=data.model_dump(exclude_unset=True)\n\n    if not cambios:\n        raise HTTPException(status_code=400,detail="No se enviaron cambios")\n\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        anterior=obtener_registro(cur,"usuarios","numero_documento_usuario",documento)\n        if not anterior:\n            raise HTTPException(status_code=404,detail="Usuario no encontrado")\n\n        campos=[]\n        valores=[]\n\n        for k,v in cambios.items():\n            campos.append(f"{k}=%s")\n            valores.append(v)\n\n        valores.append(documento)\n\n        cur.execute(\n            f"""\n            UPDATE usuarios\n            SET {\',\'.join(campos)},updated_at=now()\n            WHERE numero_documento_usuario=%s\n            RETURNING *;\n            """,\n            valores\n        )\n\n        nuevo=cur.fetchone()\n\n        registrar_auditoria(\n            cur,"usuarios",documento,"EDITAR",\n            u["numero_documento_usuario"],anterior,nuevo\n        )\n\n        db.commit()\n\n        d=dict(nuevo)\n        d.pop("password_hash",None)\n        return d\n\n    except:\n        db.rollback()\n        raise\n    finally:\n        cur.close()\n\n@app.delete("/usuarios/{documento}",tags=["Usuarios"])\ndef eliminar_usuario(documento:int,db=Depends(get_db),u=Depends(requerir_roles("Admin"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        anterior=obtener_registro(cur,"usuarios","numero_documento_usuario",documento)\n\n        if not anterior:\n            raise HTTPException(status_code=404,detail="Usuario no encontrado")\n\n        cur.execute("""\n            UPDATE usuarios\n            SET is_deleted=TRUE,\n                estado=FALSE,\n                deleted_at=now(),\n                deleted_by=%s,\n                updated_at=now()\n            WHERE numero_documento_usuario=%s\n            RETURNING *;\n        """,(u["numero_documento_usuario"],documento))\n\n        nuevo=cur.fetchone()\n\n        registrar_auditoria(\n            cur,"usuarios",documento,"ELIMINAR",\n            u["numero_documento_usuario"],anterior,nuevo\n        )\n\n        db.commit()\n        return {"mensaje":"Usuario eliminado lógicamente"}\n\n    except:\n        db.rollback()\n        raise\n    finally:\n        cur.close()\n\n@app.patch("/usuarios/{documento}/restaurar",tags=["Usuarios"])\ndef restaurar_usuario(documento:int,db=Depends(get_db),u=Depends(requerir_roles("Admin"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        anterior=obtener_registro(cur,"usuarios","numero_documento_usuario",documento)\n\n        if not anterior:\n            raise HTTPException(status_code=404,detail="Usuario no encontrado")\n\n        cur.execute("""\n            UPDATE usuarios\n            SET is_deleted=FALSE,\n                estado=TRUE,\n                deleted_at=NULL,\n                deleted_by=NULL,\n                updated_at=now()\n            WHERE numero_documento_usuario=%s\n            RETURNING *;\n        """,(documento,))\n\n        nuevo=cur.fetchone()\n\n        registrar_auditoria(\n            cur,"usuarios",documento,"RESTAURAR",\n            u["numero_documento_usuario"],anterior,nuevo\n        )\n\n        db.commit()\n        return {"mensaje":"Usuario restaurado"}\n\n    except:\n        db.rollback()\n        raise\n    finally:\n        cur.close()\n\nGeneroFHIR=Literal["male","female","other","unknown"]\nZona=Literal["urbana","rural_dispersa"]\n\nclass PacienteCreate(BaseModel):\n    numero_documento_paciente:int\n    tipo_documento:str=Field(max_length=20)\n    nombres:str=Field(max_length=100)\n    apellidos:str=Field(max_length=100)\n    fecha_nacimiento:datetime|None=None\n    genero_fhir:GeneroFHIR|None=None\n    telefono:str|None=Field(default=None,max_length=30)\n    direccion:str|None=Field(default=None,max_length=200)\n    municipio_residencia:str|None=Field(default=None,max_length=100)\n    zona_residencia:Zona|None=None\n\nclass PacienteUpdate(BaseModel):\n    tipo_documento:str|None=Field(default=None,max_length=20)\n    nombres:str|None=Field(default=None,max_length=100)\n    apellidos:str|None=Field(default=None,max_length=100)\n    fecha_nacimiento:datetime|None=None\n    genero_fhir:GeneroFHIR|None=None\n    telefono:str|None=Field(default=None,max_length=30)\n    direccion:str|None=Field(default=None,max_length=200)\n    municipio_residencia:str|None=Field(default=None,max_length=100)\n    zona_residencia:Zona|None=None\n\n@app.post("/pacientes",tags=["Pacientes"],status_code=201)\ndef crear_paciente(data:PacienteCreate,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        cur.execute("""\n            SELECT numero_documento_usuario\n            FROM usuarios\n            WHERE numero_documento_usuario=%s\n              AND id_rol=(SELECT id_rol FROM roles WHERE nombre=\'Paciente\')\n              AND is_deleted=FALSE;\n        """,(data.numero_documento_paciente,))\n\n        cuenta=cur.fetchone()\n\n        cur.execute("""\n            INSERT INTO pacientes(\n                numero_documento_paciente,id_usuario,tipo_documento,nombres,\n                apellidos,fecha_nacimiento,genero_fhir,telefono,direccion,\n                municipio_residencia,zona_residencia\n            )\n            VALUES(%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)\n            RETURNING *;\n        """,(\n            data.numero_documento_paciente,\n            cuenta["numero_documento_usuario"] if cuenta else None,\n            data.tipo_documento,data.nombres,data.apellidos,\n            data.fecha_nacimiento.date() if data.fecha_nacimiento else None,\n            data.genero_fhir,data.telefono,data.direccion,\n            data.municipio_residencia,data.zona_residencia\n        ))\n\n        nuevo=cur.fetchone()\n\n        registrar_auditoria(\n            cur,"pacientes",data.numero_documento_paciente,"CREAR",\n            u["numero_documento_usuario"],nuevos=nuevo\n        )\n\n        db.commit()\n        return nuevo\n\n    except UniqueViolation:\n        db.rollback()\n        raise HTTPException(status_code=409,detail="Paciente ya registrado")\n    except:\n        db.rollback()\n        raise\n    finally:\n        cur.close()\n\n@app.get("/pacientes",tags=["Pacientes"])\ndef listar_pacientes(db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n    try:\n        cur.execute("""\n            SELECT *\n            FROM pacientes\n            WHERE is_deleted=FALSE\n            ORDER BY apellidos,nombres;\n        """)\n        return cur.fetchall()\n    finally:\n        cur.close()\n\n@app.get("/pacientes/{documento}",tags=["Pacientes"])\ndef ver_paciente(documento:int,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico","Paciente"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        cur.execute("""\n            SELECT *\n            FROM pacientes\n            WHERE numero_documento_paciente=%s\n              AND is_deleted=FALSE;\n        """,(documento,))\n\n        p=cur.fetchone()\n\n        if not p:\n            raise HTTPException(status_code=404,detail="Paciente no encontrado")\n\n        exigir_paciente_propio(cur,documento,u)\n        return p\n\n    finally:\n        cur.close()\n\n@app.put("/pacientes/{documento}",tags=["Pacientes"])\ndef editar_paciente(documento:int,data:PacienteUpdate,db=Depends(get_db),u=Depends(requerir_roles("Admin"))):\n    cambios=data.model_dump(exclude_unset=True)\n\n    if not cambios:\n        raise HTTPException(status_code=400,detail="No se enviaron cambios")\n\n    if isinstance(cambios.get("fecha_nacimiento"),datetime):\n        cambios["fecha_nacimiento"]=cambios["fecha_nacimiento"].date()\n\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        anterior=obtener_registro(cur,"pacientes","numero_documento_paciente",documento)\n\n        if not anterior:\n            raise HTTPException(status_code=404,detail="Paciente no encontrado")\n\n        campos=[]\n        vals=[]\n\n        for k,v in cambios.items():\n            campos.append(f"{k}=%s")\n            vals.append(v)\n\n        vals.append(documento)\n\n        cur.execute(\n            f"""\n            UPDATE pacientes\n            SET {\',\'.join(campos)},updated_at=now()\n            WHERE numero_documento_paciente=%s\n            RETURNING *;\n            """,\n            vals\n        )\n\n        nuevo=cur.fetchone()\n\n        registrar_auditoria(\n            cur,"pacientes",documento,"EDITAR",\n            u["numero_documento_usuario"],anterior,nuevo\n        )\n\n        db.commit()\n        return nuevo\n\n    except:\n        db.rollback()\n        raise\n    finally:\n        cur.close()\n\n@app.delete("/pacientes/{documento}",tags=["Pacientes"])\ndef eliminar_paciente(documento:int,db=Depends(get_db),u=Depends(requerir_roles("Admin"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        anterior=obtener_registro(cur,"pacientes","numero_documento_paciente",documento)\n\n        if not anterior:\n            raise HTTPException(status_code=404,detail="Paciente no encontrado")\n\n        cur.execute("""\n            UPDATE pacientes\n            SET is_deleted=TRUE,\n                deleted_at=now(),\n                deleted_by=%s,\n                updated_at=now()\n            WHERE numero_documento_paciente=%s\n            RETURNING *;\n        """,(u["numero_documento_usuario"],documento))\n\n        nuevo=cur.fetchone()\n\n        registrar_auditoria(\n            cur,"pacientes",documento,"ELIMINAR",\n            u["numero_documento_usuario"],anterior,nuevo\n        )\n\n        db.commit()\n        return {"mensaje":"Paciente eliminado lógicamente"}\n\n    except:\n        db.rollback()\n        raise\n    finally:\n        cur.close()\n\n@app.patch("/pacientes/{documento}/restaurar",tags=["Pacientes"])\ndef restaurar_paciente(documento:int,db=Depends(get_db),u=Depends(requerir_roles("Admin"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        anterior=obtener_registro(cur,"pacientes","numero_documento_paciente",documento)\n\n        if not anterior:\n            raise HTTPException(status_code=404,detail="Paciente no encontrado")\n\n        cur.execute("""\n            UPDATE pacientes\n            SET is_deleted=FALSE,\n                deleted_at=NULL,\n                deleted_by=NULL,\n                updated_at=now()\n            WHERE numero_documento_paciente=%s\n            RETURNING *;\n        """,(documento,))\n\n        nuevo=cur.fetchone()\n\n        registrar_auditoria(\n            cur,"pacientes",documento,"RESTAURAR",\n            u["numero_documento_usuario"],anterior,nuevo\n        )\n\n        db.commit()\n        return {"mensaje":"Paciente restaurado"}\n\n    except:\n        db.rollback()\n        raise\n    finally:\n        cur.close()\n\ndef clinical_create(db,u,tabla,pk,data:dict,campo_autor="registrado_por"):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        enc=obtener_encuentro(cur,data["id_encuentro"],False)\n\n        if not enc:\n            raise HTTPException(status_code=404,detail="Encuentro no encontrado")\n\n        if enc["estado"]=="finalizado":\n            raise HTTPException(status_code=409,detail="El encuentro está finalizado")\n\n        if (\n            u["rol"]=="Medico"\n            and enc["medico_responsable"] not in (None,u["numero_documento_usuario"])\n        ):\n            raise HTTPException(status_code=403,detail="El encuentro está asignado a otro médico")\n\n        data[campo_autor]=u["numero_documento_usuario"]\n\n        cols=list(data.keys())\n        vals=[data[c] for c in cols]\n\n        cur.execute(\n            f"""\n            INSERT INTO {tabla}({\',\'.join(cols)})\n            VALUES({\',\'.join([\'%s\']*len(cols))})\n            RETURNING *;\n            """,\n            vals\n        )\n\n        nuevo=cur.fetchone()\n\n        registrar_auditoria(\n            cur,tabla,nuevo[pk],"CREAR",\n            u["numero_documento_usuario"],nuevos=nuevo\n        )\n\n        db.commit()\n        return nuevo\n\n    except:\n        db.rollback()\n        raise\n    finally:\n        cur.close()\n\ndef clinical_update(db,u,tabla,pk,id_registro,campo_autor,cambios):\n    if not cambios:\n        raise HTTPException(status_code=400,detail="No se enviaron cambios")\n\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        anterior=obtener_registro(cur,tabla,pk,id_registro)\n\n        if not anterior or anterior["is_deleted"]:\n            raise HTTPException(status_code=404,detail="Registro no encontrado")\n\n        exigir_autor_o_admin(anterior,campo_autor,u)\n\n        campos=[]\n        vals=[]\n\n        for k,v in cambios.items():\n            campos.append(f"{k}=%s")\n            vals.append(v)\n\n        vals.append(id_registro)\n\n        cur.execute(\n            f"""\n            UPDATE {tabla}\n            SET {\',\'.join(campos)},updated_at=now()\n            WHERE {pk}=%s\n            RETURNING *;\n            """,\n            vals\n        )\n\n        nuevo=cur.fetchone()\n\n        registrar_auditoria(\n            cur,tabla,id_registro,"EDITAR",\n            u["numero_documento_usuario"],anterior,nuevo\n        )\n\n        db.commit()\n        return nuevo\n\n    except:\n        db.rollback()\n        raise\n    finally:\n        cur.close()\n\ndef clinical_delete(db,u,tabla,pk,id_registro,campo_autor):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        anterior=obtener_registro(cur,tabla,pk,id_registro)\n\n        if not anterior or anterior["is_deleted"]:\n            raise HTTPException(status_code=404,detail="Registro no encontrado")\n\n        exigir_autor_o_admin(anterior,campo_autor,u)\n\n        cur.execute(\n            f"""\n            UPDATE {tabla}\n            SET is_deleted=TRUE,\n                deleted_at=now(),\n                deleted_by=%s,\n                updated_at=now()\n            WHERE {pk}=%s\n            RETURNING *;\n            """,\n            (u["numero_documento_usuario"],id_registro)\n        )\n\n        nuevo=cur.fetchone()\n\n        registrar_auditoria(\n            cur,tabla,id_registro,"ELIMINAR",\n            u["numero_documento_usuario"],anterior,nuevo\n        )\n\n        db.commit()\n        return {"mensaje":"Registro eliminado lógicamente"}\n\n    except:\n        db.rollback()\n        raise\n    finally:\n        cur.close()\n\ndef clinical_restore(db,u,tabla,pk,id_registro):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        anterior=obtener_registro(cur,tabla,pk,id_registro)\n\n        if not anterior:\n            raise HTTPException(status_code=404,detail="Registro no encontrado")\n\n        cur.execute(\n            f"""\n            UPDATE {tabla}\n            SET is_deleted=FALSE,\n                deleted_at=NULL,\n                deleted_by=NULL,\n                updated_at=now()\n            WHERE {pk}=%s\n            RETURNING *;\n            """,\n            (id_registro,)\n        )\n\n        nuevo=cur.fetchone()\n\n        registrar_auditoria(\n            cur,tabla,id_registro,"RESTAURAR",\n            u["numero_documento_usuario"],anterior,nuevo\n        )\n\n        db.commit()\n        return nuevo\n\n    except:\n        db.rollback()\n        raise\n    finally:\n        cur.close()\n\nclass AntecedenteCreate(BaseModel):\n    id_paciente:int\n    tipo:str=Field(max_length=50)\n    codigo:str|None=Field(default=None,max_length=50)\n    descripcion:str\n\nclass AntecedenteUpdate(BaseModel):\n    tipo:str|None=Field(default=None,max_length=50)\n    codigo:str|None=Field(default=None,max_length=50)\n    descripcion:str|None=None\n\n@app.post("/antecedentes",tags=["Antecedentes"],status_code=201)\ndef crear_antecedente(data:AntecedenteCreate,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        cur.execute("""\n            SELECT 1\n            FROM pacientes\n            WHERE numero_documento_paciente=%s\n              AND is_deleted=FALSE;\n        """,(data.id_paciente,))\n\n        if not cur.fetchone():\n            raise HTTPException(status_code=404,detail="Paciente no encontrado")\n\n        cur.execute("""\n            INSERT INTO antecedentes(\n                id_paciente,tipo,codigo,descripcion,registrado_por\n            )\n            VALUES(%s,%s,%s,%s,%s)\n            RETURNING *;\n        """,(\n            data.id_paciente,data.tipo,data.codigo,data.descripcion,\n            u["numero_documento_usuario"]\n        ))\n\n        nuevo=cur.fetchone()\n\n        registrar_auditoria(\n            cur,"antecedentes",nuevo["id_antecedente"],"CREAR",\n            u["numero_documento_usuario"],nuevos=nuevo\n        )\n\n        db.commit()\n        return nuevo\n\n    except:\n        db.rollback()\n        raise\n    finally:\n        cur.close()\n\n@app.get("/pacientes/{documento}/antecedentes",tags=["Antecedentes"])\ndef listar_antecedentes(documento:int,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico","Paciente"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        exigir_paciente_propio(cur,documento,u)\n\n        cur.execute("""\n            SELECT *\n            FROM antecedentes\n            WHERE id_paciente=%s\n              AND is_deleted=FALSE\n            ORDER BY fecha_registro DESC;\n        """,(documento,))\n\n        return cur.fetchall()\n\n    finally:\n        cur.close()\n\n@app.put("/antecedentes/{id_antecedente}",tags=["Antecedentes"])\ndef editar_antecedente(id_antecedente:int,data:AntecedenteUpdate,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    return clinical_update(\n        db,u,"antecedentes","id_antecedente",\n        id_antecedente,"registrado_por",\n        data.model_dump(exclude_unset=True)\n    )\n\n@app.delete("/antecedentes/{id_antecedente}",tags=["Antecedentes"])\ndef eliminar_antecedente(id_antecedente:int,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    return clinical_delete(\n        db,u,"antecedentes","id_antecedente",\n        id_antecedente,"registrado_por"\n    )\n\n@app.patch("/antecedentes/{id_antecedente}/restaurar",tags=["Antecedentes"])\ndef restaurar_antecedente(id_antecedente:int,db=Depends(get_db),u=Depends(requerir_roles("Admin"))):\n    return clinical_restore(\n        db,u,"antecedentes","id_antecedente",id_antecedente\n    )\n\nclass ReporteCreate(BaseModel):\n    id_paciente:int\n    sintoma_principal:str\n    inicio_sintomas:datetime|None=None\n    evolucion:str|None=None\n    signos_alarma_presentes:bool=False\n    descripcion_signos_alarma:str|None=None\n    ubicacion_aproximada:str|None=None\n    municipio_origen:str|None=None\n    distancia_aproximada_km:Decimal|None=Field(default=None,ge=0)\n    tiempo_desplazamiento_min:int|None=Field(default=None,ge=0)\n\n@app.post("/reportes-previos",tags=["Reportes previos"],status_code=201)\ndef crear_reporte(data:ReporteCreate,db=Depends(get_db),u=Depends(requerir_roles("Paciente","Admin"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        if u["rol"]=="Paciente":\n            exigir_paciente_propio(cur,data.id_paciente,u)\n\n        if data.signos_alarma_presentes and not data.descripcion_signos_alarma:\n            raise HTTPException(status_code=400,detail="Debe describir los signos de alarma")\n\n        d=data.model_dump()\n        cols=list(d.keys())+["registrado_por"]\n        vals=[d[k] for k in d]+[u["numero_documento_usuario"]]\n\n        cur.execute(\n            f"""\n            INSERT INTO reportes_previos({\',\'.join(cols)})\n            VALUES({\',\'.join([\'%s\']*len(cols))})\n            RETURNING *;\n            """,\n            vals\n        )\n\n        nuevo=cur.fetchone()\n\n        registrar_auditoria(\n            cur,"reportes_previos",nuevo["id_reporte"],"CREAR",\n            u["numero_documento_usuario"],nuevos=nuevo\n        )\n\n        db.commit()\n        return nuevo\n\n    except:\n        db.rollback()\n        raise\n    finally:\n        cur.close()\n\n@app.get("/reportes-previos",tags=["Reportes previos"])\ndef listar_reportes(db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        cur.execute("""\n            SELECT *\n            FROM reportes_previos\n            WHERE is_deleted=FALSE\n            ORDER BY fecha_hora_reporte DESC;\n        """)\n        return cur.fetchall()\n    finally:\n        cur.close()\n\n@app.get("/reportes-previos/mios",tags=["Reportes previos"])\ndef mis_reportes(db=Depends(get_db),u=Depends(requerir_roles("Paciente"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        cur.execute("""\n            SELECT rp.*\n            FROM reportes_previos rp\n            JOIN pacientes p\n              ON p.numero_documento_paciente=rp.id_paciente\n            WHERE p.id_usuario=%s\n              AND rp.is_deleted=FALSE\n            ORDER BY rp.fecha_hora_reporte DESC;\n        """,(u["numero_documento_usuario"],))\n\n        return cur.fetchall()\n\n    finally:\n        cur.close()\n\nclass EncuentroCreate(BaseModel):\n    id_paciente:int\n    motivo_consulta:str\n    tipo_encuentro:str="urgencias"\n    servicio:str="URGENCIAS"\n    medico_responsable:int|None=None\n\nclass TriageUpdate(BaseModel):\n    nivel_triage:int=Field(ge=1,le=5)\n    dolor_escala:int|None=Field(default=None,ge=0,le=10)\n    observaciones_triage:str|None=None\n\n@app.post("/encuentros",tags=["Encuentros"],status_code=201)\ndef crear_encuentro(data:EncuentroCreate,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        cur.execute("""\n            SELECT 1\n            FROM pacientes\n            WHERE numero_documento_paciente=%s\n              AND is_deleted=FALSE;\n        """,(data.id_paciente,))\n\n        if not cur.fetchone():\n            raise HTTPException(status_code=404,detail="Paciente no encontrado")\n\n        medico=data.medico_responsable\n\n        if u["rol"]=="Medico":\n            medico=u["numero_documento_usuario"]\n\n        cur.execute("""\n            INSERT INTO encuentros(\n                id_paciente,tipo_encuentro,servicio,motivo_consulta,\n                medico_responsable,creado_por\n            )\n            VALUES(%s,%s,%s,%s,%s,%s)\n            RETURNING *;\n        """,(\n            data.id_paciente,data.tipo_encuentro,data.servicio,\n            data.motivo_consulta,medico,u["numero_documento_usuario"]\n        ))\n\n        nuevo=cur.fetchone()\n\n        registrar_auditoria(\n            cur,"encuentros",nuevo["id_encuentro"],"CREAR",\n            u["numero_documento_usuario"],nuevos=nuevo\n        )\n\n        db.commit()\n        return nuevo\n\n    except UniqueViolation:\n        db.rollback()\n        raise HTTPException(status_code=409,detail="El paciente ya tiene un encuentro activo")\n    except:\n        db.rollback()\n        raise\n    finally:\n        cur.close()\n\n@app.get("/encuentros",tags=["Encuentros"])\ndef listar_encuentros(db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        cur.execute("""\n            SELECT *\n            FROM encuentros\n            WHERE is_deleted=FALSE\n            ORDER BY fecha_hora_ingreso DESC;\n        """)\n        return cur.fetchall()\n    finally:\n        cur.close()\n\n@app.get("/encuentros/mios",tags=["Encuentros"])\ndef mis_encuentros(db=Depends(get_db),u=Depends(requerir_roles("Paciente"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        cur.execute("""\n            SELECT e.*\n            FROM encuentros e\n            JOIN pacientes p\n              ON p.numero_documento_paciente=e.id_paciente\n            WHERE p.id_usuario=%s\n              AND e.is_deleted=FALSE\n            ORDER BY e.fecha_hora_ingreso DESC;\n        """,(u["numero_documento_usuario"],))\n\n        return cur.fetchall()\n    finally:\n        cur.close()\n\n@app.get("/encuentros/{id_encuentro}",tags=["Encuentros"])\ndef ver_encuentro(id_encuentro:int,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico","Paciente"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        e=obtener_encuentro(cur,id_encuentro,False)\n        exigir_acceso_encuentro(cur,e,u)\n        return e\n    finally:\n        cur.close()\n\n@app.put("/encuentros/{id_encuentro}/triage",tags=["Encuentros"])\ndef registrar_triage(id_encuentro:int,data:TriageUpdate,db=Depends(get_db),u=Depends(requerir_roles("Medico"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        anterior=obtener_encuentro(cur,id_encuentro,False)\n\n        if not anterior:\n            raise HTTPException(status_code=404,detail="Encuentro no encontrado")\n\n        if anterior["medico_responsable"] not in (None,u["numero_documento_usuario"]):\n            raise HTTPException(status_code=403,detail="El encuentro está asignado a otro médico")\n\n        if anterior["estado"]=="finalizado":\n            raise HTTPException(status_code=409,detail="Encuentro finalizado")\n\n        cur.execute("""\n            UPDATE encuentros\n            SET nivel_triage=%s,\n                dolor_escala=%s,\n                observaciones_triage=%s,\n                fecha_hora_triage=now(),\n                clasificado_por=%s,\n                medico_responsable=COALESCE(medico_responsable,%s),\n                estado=\'en_atencion\',\n                updated_at=now()\n            WHERE id_encuentro=%s\n            RETURNING *;\n        """,(\n            data.nivel_triage,data.dolor_escala,\n            data.observaciones_triage,\n            u["numero_documento_usuario"],\n            u["numero_documento_usuario"],\n            id_encuentro\n        ))\n\n        nuevo=cur.fetchone()\n\n        registrar_auditoria(\n            cur,"encuentros",id_encuentro,"EDITAR",\n            u["numero_documento_usuario"],anterior,nuevo\n        )\n\n        db.commit()\n        return nuevo\n\n    except:\n        db.rollback()\n        raise\n    finally:\n        cur.close()\n\n@app.patch("/encuentros/{id_encuentro}/finalizar",tags=["Encuentros"])\ndef finalizar_encuentro(id_encuentro:int,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        anterior=obtener_encuentro(cur,id_encuentro,False)\n\n        if not anterior:\n            raise HTTPException(status_code=404,detail="Encuentro no encontrado")\n\n        if (\n            u["rol"]=="Medico"\n            and anterior["medico_responsable"]!=u["numero_documento_usuario"]\n        ):\n            raise HTTPException(status_code=403,detail="Solo el médico responsable puede finalizar el encuentro")\n\n        cur.execute("""\n            UPDATE encuentros\n            SET estado=\'finalizado\',\n                fecha_hora_fin=now(),\n                updated_at=now()\n            WHERE id_encuentro=%s\n            RETURNING *;\n        """,(id_encuentro,))\n\n        nuevo=cur.fetchone()\n\n        registrar_auditoria(\n            cur,"encuentros",id_encuentro,"FINALIZAR",\n            u["numero_documento_usuario"],anterior,nuevo\n        )\n\n        db.commit()\n        return nuevo\n\n    except:\n        db.rollback()\n        raise\n    finally:\n        cur.close()\n\nclass ObservacionCreate(BaseModel):\n    id_encuentro:int\n    tipo_observacion:str=Field(max_length=100)\n    codigo_loinc:str|None=Field(default=None,max_length=30)\n    nombre:str=Field(max_length=150)\n    valor_numerico:Decimal|None=None\n    valor_texto:str|None=None\n    unidad:str|None=Field(default=None,max_length=30)\n\nclass ObservacionUpdate(BaseModel):\n    tipo_observacion:str|None=Field(default=None,max_length=100)\n    codigo_loinc:str|None=Field(default=None,max_length=30)\n    nombre:str|None=Field(default=None,max_length=150)\n    valor_numerico:Decimal|None=None\n    valor_texto:str|None=None\n    unidad:str|None=Field(default=None,max_length=30)\n\ndef validar_observacion(d):\n    if (d.get("valor_numerico") is None)==(d.get("valor_texto") is None):\n        raise HTTPException(\n            status_code=400,\n            detail="Debe enviar exactamente uno entre valor_numerico y valor_texto"\n        )\n\n@app.post("/observaciones",tags=["Observaciones"],status_code=201)\ndef crear_observacion(data:ObservacionCreate,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    d=data.model_dump()\n    validar_observacion(d)\n    return clinical_create(db,u,"observaciones","id_observacion",d)\n\n@app.get("/encuentros/{id_encuentro}/observaciones",tags=["Observaciones"])\ndef observaciones_encuentro(id_encuentro:int,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico","Paciente"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        e=obtener_encuentro(cur,id_encuentro,False)\n        exigir_acceso_encuentro(cur,e,u)\n\n        cur.execute("""\n            SELECT *\n            FROM observaciones\n            WHERE id_encuentro=%s\n              AND is_deleted=FALSE\n            ORDER BY fecha_hora_observacion;\n        """,(id_encuentro,))\n\n        return cur.fetchall()\n\n    finally:\n        cur.close()\n\n@app.put("/observaciones/{id_observacion}",tags=["Observaciones"])\ndef editar_observacion(id_observacion:int,data:ObservacionUpdate,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    cambios=data.model_dump(exclude_unset=True)\n\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        actual=obtener_registro(cur,"observaciones","id_observacion",id_observacion)\n\n        if not actual:\n            raise HTTPException(status_code=404,detail="Observación no encontrada")\n\n        combinado=dict(actual)\n        combinado.update(cambios)\n        validar_observacion(combinado)\n\n    finally:\n        cur.close()\n\n    return clinical_update(\n        db,u,"observaciones","id_observacion",\n        id_observacion,"registrado_por",cambios\n    )\n\n@app.delete("/observaciones/{id_observacion}",tags=["Observaciones"])\ndef eliminar_observacion(id_observacion:int,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    return clinical_delete(\n        db,u,"observaciones","id_observacion",\n        id_observacion,"registrado_por"\n    )\n\n@app.patch("/observaciones/{id_observacion}/restaurar",tags=["Observaciones"])\ndef restaurar_observacion(id_observacion:int,db=Depends(get_db),u=Depends(requerir_roles("Admin"))):\n    return clinical_restore(\n        db,u,"observaciones","id_observacion",id_observacion\n    )\n\nclass DiagnosticoCreate(BaseModel):\n    id_encuentro:int\n    codigo_cie10:str|None=Field(default=None,max_length=30)\n    descripcion:str=Field(max_length=250)\n    tipo:Literal["principal","secundario"]="principal"\n    estado_clinico:Literal[\n        "active","recurrence","relapse",\n        "inactive","remission","resolved"\n    ]="active"\n\nclass DiagnosticoUpdate(BaseModel):\n    codigo_cie10:str|None=Field(default=None,max_length=30)\n    descripcion:str|None=Field(default=None,max_length=250)\n    tipo:Literal["principal","secundario"]|None=None\n    estado_clinico:Literal[\n        "active","recurrence","relapse",\n        "inactive","remission","resolved"\n    ]|None=None\n\n@app.post("/diagnosticos",tags=["Diagnósticos"],status_code=201)\ndef crear_diagnostico(data:DiagnosticoCreate,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    return clinical_create(\n        db,u,"diagnosticos","id_diagnostico",data.model_dump()\n    )\n\n@app.get("/encuentros/{id_encuentro}/diagnosticos",tags=["Diagnósticos"])\ndef diagnosticos_encuentro(id_encuentro:int,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico","Paciente"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        e=obtener_encuentro(cur,id_encuentro,False)\n        exigir_acceso_encuentro(cur,e,u)\n\n        cur.execute("""\n            SELECT *\n            FROM diagnosticos\n            WHERE id_encuentro=%s\n              AND is_deleted=FALSE\n            ORDER BY fecha_diagnostico;\n        """,(id_encuentro,))\n\n        return cur.fetchall()\n\n    finally:\n        cur.close()\n\n@app.put("/diagnosticos/{id_diagnostico}",tags=["Diagnósticos"])\ndef editar_diagnostico(id_diagnostico:int,data:DiagnosticoUpdate,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    return clinical_update(\n        db,u,"diagnosticos","id_diagnostico",\n        id_diagnostico,"registrado_por",\n        data.model_dump(exclude_unset=True)\n    )\n\n@app.delete("/diagnosticos/{id_diagnostico}",tags=["Diagnósticos"])\ndef eliminar_diagnostico(id_diagnostico:int,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    return clinical_delete(\n        db,u,"diagnosticos","id_diagnostico",\n        id_diagnostico,"registrado_por"\n    )\n\n@app.patch("/diagnosticos/{id_diagnostico}/restaurar",tags=["Diagnósticos"])\ndef restaurar_diagnostico(id_diagnostico:int,db=Depends(get_db),u=Depends(requerir_roles("Admin"))):\n    return clinical_restore(\n        db,u,"diagnosticos","id_diagnostico",id_diagnostico\n    )\n\nclass NotaCreate(BaseModel):\n    id_encuentro:int\n    tipo_nota:Literal["evolucion","valoracion","nota_clinica"]\n    contenido:str\n\nclass NotaUpdate(BaseModel):\n    tipo_nota:Literal["evolucion","valoracion","nota_clinica"]|None=None\n    contenido:str|None=None\n\n@app.post("/notas-clinicas",tags=["Notas clínicas"],status_code=201)\ndef crear_nota(data:NotaCreate,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    return clinical_create(\n        db,u,"notas_clinicas","id_nota",data.model_dump()\n    )\n\n@app.get("/encuentros/{id_encuentro}/notas-clinicas",tags=["Notas clínicas"])\ndef notas_encuentro(id_encuentro:int,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico","Paciente"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        e=obtener_encuentro(cur,id_encuentro,False)\n        exigir_acceso_encuentro(cur,e,u)\n\n        cur.execute("""\n            SELECT *\n            FROM notas_clinicas\n            WHERE id_encuentro=%s\n              AND is_deleted=FALSE\n            ORDER BY fecha_hora;\n        """,(id_encuentro,))\n\n        return cur.fetchall()\n\n    finally:\n        cur.close()\n\n@app.put("/notas-clinicas/{id_nota}",tags=["Notas clínicas"])\ndef editar_nota(id_nota:int,data:NotaUpdate,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    return clinical_update(\n        db,u,"notas_clinicas","id_nota",\n        id_nota,"registrado_por",\n        data.model_dump(exclude_unset=True)\n    )\n\n@app.delete("/notas-clinicas/{id_nota}",tags=["Notas clínicas"])\ndef eliminar_nota(id_nota:int,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    return clinical_delete(\n        db,u,"notas_clinicas","id_nota",\n        id_nota,"registrado_por"\n    )\n\n@app.patch("/notas-clinicas/{id_nota}/restaurar",tags=["Notas clínicas"])\ndef restaurar_nota(id_nota:int,db=Depends(get_db),u=Depends(requerir_roles("Admin"))):\n    return clinical_restore(\n        db,u,"notas_clinicas","id_nota",id_nota\n    )\n\nEstadoExamen=Literal[\n    "draft","active","on-hold","revoked",\n    "completed","entered-in-error","unknown"\n]\n\nclass ExamenCreate(BaseModel):\n    id_encuentro:int\n    codigo_loinc:str|None=Field(default=None,max_length=30)\n    nombre:str=Field(max_length=150)\n    categoria:str|None=Field(default=None,max_length=100)\n\nclass ExamenUpdate(BaseModel):\n    codigo_loinc:str|None=Field(default=None,max_length=30)\n    nombre:str|None=Field(default=None,max_length=150)\n    categoria:str|None=Field(default=None,max_length=100)\n    estado:EstadoExamen|None=None\n    resultado:str|None=None\n    conclusion:str|None=None\n    fecha_resultado:datetime|None=None\n\n@app.post("/examenes",tags=["Exámenes"],status_code=201)\ndef crear_examen(data:ExamenCreate,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    d=data.model_dump()\n    d["estado"]="active"\n    return clinical_create(\n        db,u,"examenes","id_examen",d,"solicitado_por"\n    )\n\n@app.get("/encuentros/{id_encuentro}/examenes",tags=["Exámenes"])\ndef examenes_encuentro(id_encuentro:int,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico","Paciente"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        e=obtener_encuentro(cur,id_encuentro,False)\n        exigir_acceso_encuentro(cur,e,u)\n\n        cur.execute("""\n            SELECT *\n            FROM examenes\n            WHERE id_encuentro=%s\n              AND is_deleted=FALSE\n            ORDER BY fecha_solicitud;\n        """,(id_encuentro,))\n\n        return cur.fetchall()\n\n    finally:\n        cur.close()\n\n@app.put("/examenes/{id_examen}",tags=["Exámenes"])\ndef editar_examen(id_examen:int,data:ExamenUpdate,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    cambios=data.model_dump(exclude_unset=True)\n\n    if cambios.get("estado")=="completed":\n        cambios["registrado_resultado_por"]=u["numero_documento_usuario"]\n\n        if cambios.get("fecha_resultado") is None:\n            cambios["fecha_resultado"]=datetime.now(timezone.utc)\n\n    return clinical_update(\n        db,u,"examenes","id_examen",\n        id_examen,"solicitado_por",cambios\n    )\n\n@app.delete("/examenes/{id_examen}",tags=["Exámenes"])\ndef eliminar_examen(id_examen:int,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    return clinical_delete(\n        db,u,"examenes","id_examen",\n        id_examen,"solicitado_por"\n    )\n\n@app.patch("/examenes/{id_examen}/restaurar",tags=["Exámenes"])\ndef restaurar_examen(id_examen:int,db=Depends(get_db),u=Depends(requerir_roles("Admin"))):\n    return clinical_restore(\n        db,u,"examenes","id_examen",id_examen\n    )\n\nclass MedicamentoCreate(BaseModel):\n    codigo_cum:str=Field(max_length=50)\n    nombre:str=Field(max_length=150)\n    principio_activo:str|None=Field(default=None,max_length=150)\n    concentracion:str|None=Field(default=None,max_length=100)\n    forma_farmaceutica:str|None=Field(default=None,max_length=100)\n    registro_sanitario:str|None=Field(default=None,max_length=100)\n    estado_cum:str|None=Field(default=None,max_length=30)\n    precio_unitario:Decimal=Field(ge=0)\n\n@app.post("/medicamentos",tags=["Medicamentos"],status_code=201)\ndef crear_medicamento(data:MedicamentoCreate,db=Depends(get_db),u=Depends(requerir_roles("Admin"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        d=data.model_dump()\n        cols=list(d)\n        vals=[d[k] for k in cols]\n\n        cur.execute(\n            f"""\n            INSERT INTO medicamentos({\',\'.join(cols)})\n            VALUES({\',\'.join([\'%s\']*len(cols))})\n            RETURNING *;\n            """,\n            vals\n        )\n\n        nuevo=cur.fetchone()\n\n        registrar_auditoria(\n            cur,"medicamentos",nuevo["codigo_cum"],"CREAR",\n            u["numero_documento_usuario"],nuevos=nuevo\n        )\n\n        db.commit()\n        return nuevo\n\n    except UniqueViolation:\n        db.rollback()\n        raise HTTPException(status_code=409,detail="Medicamento ya existe")\n    except:\n        db.rollback()\n        raise\n    finally:\n        cur.close()\n\n@app.get("/medicamentos",tags=["Medicamentos"])\ndef listar_medicamentos(db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        cur.execute("""\n            SELECT *\n            FROM medicamentos\n            WHERE is_deleted=FALSE\n            ORDER BY nombre;\n        """)\n        return cur.fetchall()\n    finally:\n        cur.close()\n\nclass PrescripcionCreate(BaseModel):\n    id_encuentro:int\n    codigo_cum:str=Field(max_length=50)\n    dosis:str|None=Field(default=None,max_length=100)\n    frecuencia:str|None=Field(default=None,max_length=100)\n    via_administracion:str|None=Field(default=None,max_length=50)\n    cantidad:int=Field(gt=0)\n\nclass PrescripcionUpdate(BaseModel):\n    dosis:str|None=Field(default=None,max_length=100)\n    frecuencia:str|None=Field(default=None,max_length=100)\n    via_administracion:str|None=Field(default=None,max_length=50)\n    cantidad:int|None=Field(default=None,gt=0)\n\n@app.post("/prescripciones",tags=["Prescripciones"],status_code=201)\ndef crear_prescripcion(data:PrescripcionCreate,db=Depends(get_db),u=Depends(requerir_roles("Medico"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        cur.execute("""\n            SELECT 1\n            FROM medicamentos\n            WHERE codigo_cum=%s\n              AND is_deleted=FALSE;\n        """,(data.codigo_cum,))\n\n        if not cur.fetchone():\n            raise HTTPException(status_code=404,detail="Medicamento no encontrado")\n\n    finally:\n        cur.close()\n\n    return clinical_create(\n        db,u,"prescripciones","id_prescripcion",\n        data.model_dump(),"prescrito_por"\n    )\n\n@app.get("/prescripciones",tags=["Prescripciones"])\ndef listar_prescripciones(db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        cur.execute("""\n            SELECT\n                pr.*,m.nombre AS medicamento,e.id_paciente\n            FROM prescripciones pr\n            JOIN medicamentos m ON m.codigo_cum=pr.codigo_cum\n            JOIN encuentros e ON e.id_encuentro=pr.id_encuentro\n            WHERE pr.is_deleted=FALSE\n            ORDER BY pr.fecha_prescripcion DESC;\n        """)\n        return cur.fetchall()\n\n    finally:\n        cur.close()\n\n@app.get("/encuentros/{id_encuentro}/prescripciones",tags=["Prescripciones"])\ndef prescripciones_encuentro(id_encuentro:int,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico","Paciente"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        e=obtener_encuentro(cur,id_encuentro,False)\n        exigir_acceso_encuentro(cur,e,u)\n\n        cur.execute("""\n            SELECT\n                pr.*,m.nombre AS medicamento\n            FROM prescripciones pr\n            JOIN medicamentos m ON m.codigo_cum=pr.codigo_cum\n            WHERE pr.id_encuentro=%s\n              AND pr.is_deleted=FALSE\n            ORDER BY pr.fecha_prescripcion;\n        """,(id_encuentro,))\n\n        return cur.fetchall()\n\n    finally:\n        cur.close()\n\n@app.put("/prescripciones/{id_prescripcion}",tags=["Prescripciones"])\ndef editar_prescripcion(id_prescripcion:int,data:PrescripcionUpdate,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    return clinical_update(\n        db,u,"prescripciones","id_prescripcion",\n        id_prescripcion,"prescrito_por",\n        data.model_dump(exclude_unset=True)\n    )\n\ndef cambiar_estado_prescripcion(id_prescripcion,nuevo_estado,accion,db,u):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        anterior=obtener_registro(cur,"prescripciones","id_prescripcion",id_prescripcion)\n\n        if not anterior or anterior["is_deleted"]:\n            raise HTTPException(status_code=404,detail="Prescripción no encontrada")\n\n        exigir_autor_o_admin(anterior,"prescrito_por",u)\n\n        cur.execute("""\n            UPDATE prescripciones\n            SET estado=%s,\n                updated_at=now()\n            WHERE id_prescripcion=%s\n            RETURNING *;\n        """,(nuevo_estado,id_prescripcion))\n\n        nuevo=cur.fetchone()\n\n        registrar_auditoria(\n            cur,"prescripciones",id_prescripcion,accion,\n            u["numero_documento_usuario"],anterior,nuevo\n        )\n\n        db.commit()\n        return nuevo\n\n    except:\n        db.rollback()\n        raise\n    finally:\n        cur.close()\n\n@app.patch("/prescripciones/{id_prescripcion}/dispensar",tags=["Prescripciones"])\ndef dispensar_prescripcion(id_prescripcion:int,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    return cambiar_estado_prescripcion(\n        id_prescripcion,"dispensada","DISPENSAR",db,u\n    )\n\n@app.patch("/prescripciones/{id_prescripcion}/anular",tags=["Prescripciones"])\ndef anular_prescripcion(id_prescripcion:int,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    return cambiar_estado_prescripcion(\n        id_prescripcion,"anulada","ANULAR",db,u\n    )\n\n@app.delete("/prescripciones/{id_prescripcion}",tags=["Prescripciones"])\ndef eliminar_prescripcion(id_prescripcion:int,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico"))):\n    return clinical_delete(\n        db,u,"prescripciones","id_prescripcion",\n        id_prescripcion,"prescrito_por"\n    )\n\n@app.patch("/prescripciones/{id_prescripcion}/restaurar",tags=["Prescripciones"])\ndef restaurar_prescripcion(id_prescripcion:int,db=Depends(get_db),u=Depends(requerir_roles("Admin"))):\n    return clinical_restore(\n        db,u,"prescripciones","id_prescripcion",id_prescripcion\n    )\n\nclass DetalleAdicional(BaseModel):\n    concepto:str=Field(max_length=200)\n    cantidad:int=Field(default=1,gt=0)\n    valor_unitario:Decimal=Field(ge=0)\n\nclass FacturaCreate(BaseModel):\n    id_encuentro:int\n    numero_factura:str=Field(max_length=50)\n    concepto:str|None=None\n    incluir_prescripciones_dispensadas:bool=True\n    incluir_examenes_completados:bool=True\n    detalles_adicionales:list[DetalleAdicional]=Field(default_factory=list)\n\n@app.post("/facturas",tags=["Facturación"],status_code=201)\ndef crear_factura(data:FacturaCreate,db=Depends(get_db),u=Depends(requerir_roles("Admin","Administrativo"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        e=obtener_encuentro(cur,data.id_encuentro,False)\n\n        if not e:\n            raise HTTPException(status_code=404,detail="Encuentro no encontrado")\n\n        if e["estado"]!="finalizado":\n            raise HTTPException(status_code=409,detail="Solo se factura un encuentro finalizado")\n\n        cur.execute("""\n            INSERT INTO facturas(\n                id_paciente,id_encuentro,numero_factura,\n                concepto,total,creado_por\n            )\n            VALUES(%s,%s,%s,%s,0,%s)\n            RETURNING *;\n        """,(\n            e["id_paciente"],e["id_encuentro"],\n            data.numero_factura,data.concepto,\n            u["numero_documento_usuario"]\n        ))\n\n        f=cur.fetchone()\n        detalles=[]\n\n        if data.incluir_prescripciones_dispensadas:\n            cur.execute("""\n                SELECT\n                    pr.id_prescripcion,pr.cantidad,\n                    m.nombre,m.precio_unitario\n                FROM prescripciones pr\n                JOIN medicamentos m ON m.codigo_cum=pr.codigo_cum\n                WHERE pr.id_encuentro=%s\n                  AND pr.estado=\'dispensada\'\n                  AND pr.is_deleted=FALSE;\n            """,(e["id_encuentro"],))\n\n            for pr in cur.fetchall():\n                detalles.append((\n                    pr["id_prescripcion"],None,\n                    f"Medicamento: {pr[\'nombre\']}",\n                    pr["cantidad"],pr["precio_unitario"]\n                ))\n\n        if data.incluir_examenes_completados:\n            cur.execute("""\n                SELECT id_examen,nombre\n                FROM examenes\n                WHERE id_encuentro=%s\n                  AND estado=\'completed\'\n                  AND is_deleted=FALSE;\n            """,(e["id_encuentro"],))\n\n            for ex in cur.fetchall():\n                detalles.append((\n                    None,ex["id_examen"],\n                    f"Examen: {ex[\'nombre\']}",\n                    1,Decimal("0")\n                ))\n\n        for d in data.detalles_adicionales:\n            detalles.append((\n                None,None,d.concepto,\n                d.cantidad,d.valor_unitario\n            ))\n\n        total=Decimal("0")\n\n        for idp,idx,concepto,cantidad,valor in detalles:\n            vt=Decimal(cantidad)*Decimal(valor)\n            total+=vt\n\n            cur.execute("""\n                INSERT INTO factura_detalle(\n                    id_factura,id_encuentro,id_prescripcion,id_examen,\n                    concepto,cantidad,valor_unitario,valor_total,creado_por\n                )\n                VALUES(%s,%s,%s,%s,%s,%s,%s,%s,%s);\n            """,(\n                f["id_factura"],e["id_encuentro"],\n                idp,idx,concepto,cantidad,\n                valor,vt,u["numero_documento_usuario"]\n            ))\n\n        cur.execute("""\n            UPDATE facturas\n            SET total=%s,\n                updated_at=now()\n            WHERE id_factura=%s\n            RETURNING *;\n        """,(total,f["id_factura"]))\n\n        nuevo=cur.fetchone()\n\n        registrar_auditoria(\n            cur,"facturas",nuevo["id_factura"],"CREAR",\n            u["numero_documento_usuario"],nuevos=nuevo\n        )\n\n        db.commit()\n        return nuevo\n\n    except UniqueViolation:\n        db.rollback()\n        raise HTTPException(\n            status_code=409,\n            detail="Ya existe una factura activa para el encuentro o el número está repetido"\n        )\n    except:\n        db.rollback()\n        raise\n    finally:\n        cur.close()\n\n@app.get("/facturas",tags=["Facturación"])\ndef listar_facturas(db=Depends(get_db),u=Depends(requerir_roles("Admin","Administrativo"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        cur.execute("""\n            SELECT *\n            FROM facturas\n            WHERE is_deleted=FALSE\n            ORDER BY fecha_emision DESC;\n        """)\n        return cur.fetchall()\n    finally:\n        cur.close()\n\n@app.get("/facturas/{id_factura}",tags=["Facturación"])\ndef ver_factura(id_factura:int,db=Depends(get_db),u=Depends(requerir_roles("Admin","Administrativo"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        cur.execute("""\n            SELECT *\n            FROM facturas\n            WHERE id_factura=%s\n              AND is_deleted=FALSE;\n        """,(id_factura,))\n\n        f=cur.fetchone()\n\n        if not f:\n            raise HTTPException(status_code=404,detail="Factura no encontrada")\n\n        cur.execute("""\n            SELECT *\n            FROM factura_detalle\n            WHERE id_factura=%s\n              AND is_deleted=FALSE\n            ORDER BY id_detalle;\n        """,(id_factura,))\n\n        d=dict(f)\n        d["detalles"]=cur.fetchall()\n        return d\n\n    finally:\n        cur.close()\n\n@app.get("/pacientes/{documento}/historia-clinica",tags=["Historia clínica"])\ndef historia_clinica(documento:int,db=Depends(get_db),u=Depends(requerir_roles("Admin","Medico","Paciente"))):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        cur.execute("""\n            SELECT *\n            FROM pacientes\n            WHERE numero_documento_paciente=%s\n              AND is_deleted=FALSE;\n        """,(documento,))\n\n        p=cur.fetchone()\n\n        if not p:\n            raise HTTPException(status_code=404,detail="Paciente no encontrado")\n\n        exigir_paciente_propio(cur,documento,u)\n\n        cur.execute("""\n            SELECT *\n            FROM antecedentes\n            WHERE id_paciente=%s\n              AND is_deleted=FALSE\n            ORDER BY fecha_registro DESC;\n        """,(documento,))\n\n        antecedentes=cur.fetchall()\n\n        cur.execute("""\n            SELECT *\n            FROM encuentros\n            WHERE id_paciente=%s\n              AND is_deleted=FALSE\n            ORDER BY fecha_hora_ingreso DESC;\n        """,(documento,))\n\n        encuentros=cur.fetchall()\n        episodios=[]\n\n        for e in encuentros:\n            ide=e["id_encuentro"]\n\n            cur.execute("""\n                SELECT *\n                FROM observaciones\n                WHERE id_encuentro=%s\n                  AND is_deleted=FALSE\n                ORDER BY fecha_hora_observacion;\n            """,(ide,))\n            obs=cur.fetchall()\n\n            cur.execute("""\n                SELECT *\n                FROM diagnosticos\n                WHERE id_encuentro=%s\n                  AND is_deleted=FALSE\n                ORDER BY fecha_diagnostico;\n            """,(ide,))\n            dx=cur.fetchall()\n\n            cur.execute("""\n                SELECT *\n                FROM notas_clinicas\n                WHERE id_encuentro=%s\n                  AND is_deleted=FALSE\n                ORDER BY fecha_hora;\n            """,(ide,))\n            notas=cur.fetchall()\n\n            cur.execute("""\n                SELECT *\n                FROM examenes\n                WHERE id_encuentro=%s\n                  AND is_deleted=FALSE\n                ORDER BY fecha_solicitud;\n            """,(ide,))\n            exam=cur.fetchall()\n\n            cur.execute("""\n                SELECT\n                    pr.*,m.nombre AS medicamento,\n                    m.principio_activo,m.concentracion,\n                    m.forma_farmaceutica\n                FROM prescripciones pr\n                JOIN medicamentos m\n                  ON m.codigo_cum=pr.codigo_cum\n                WHERE pr.id_encuentro=%s\n                  AND pr.is_deleted=FALSE\n                ORDER BY pr.fecha_prescripcion;\n            """,(ide,))\n            pres=cur.fetchall()\n\n            ep=dict(e)\n\n            ep["triage"]={\n                "nivel_triage":e["nivel_triage"],\n                "fecha_hora_triage":e["fecha_hora_triage"],\n                "dolor_escala":e["dolor_escala"],\n                "observaciones_triage":e["observaciones_triage"],\n                "clasificado_por":e["clasificado_por"]\n            }\n\n            ep["observaciones"]=obs\n            ep["diagnosticos"]=dx\n            ep["notas_clinicas"]=notas\n            ep["examenes"]=exam\n            ep["prescripciones"]=pres\n\n            episodios.append(ep)\n\n        return {\n            "paciente":p,\n            "antecedentes":antecedentes,\n            "encuentros":episodios\n        }\n\n    finally:\n        cur.close()\n\nTablaAuditoria=Literal[\n    "usuarios","pacientes","antecedentes","reportes_previos",\n    "encuentros","observaciones","diagnosticos","notas_clinicas",\n    "examenes","medicamentos","prescripciones","facturas",\n    "factura_detalle"\n]\n\n@app.get("/auditoria",tags=["Auditoría"])\ndef auditoria(\n    limite:int=Query(100,ge=1,le=500),\n    db=Depends(get_db),\n    u=Depends(requerir_roles("Admin"))\n):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        cur.execute("""\n            SELECT\n                a.*,u.username,r.nombre AS rol\n            FROM auditoria_cambios a\n            LEFT JOIN usuarios u\n              ON u.numero_documento_usuario=a.realizado_por\n            LEFT JOIN roles r\n              ON r.id_rol=u.id_rol\n            ORDER BY a.fecha_hora DESC\n            LIMIT %s;\n        """,(limite,))\n\n        return cur.fetchall()\n\n    finally:\n        cur.close()\n\n@app.get("/auditoria/{tabla}/{registro_id}",tags=["Auditoría"])\ndef auditoria_registro(\n    tabla:TablaAuditoria,\n    registro_id:str,\n    db=Depends(get_db),\n    u=Depends(requerir_roles("Admin"))\n):\n    cur=db.cursor(cursor_factory=RealDictCursor)\n\n    try:\n        cur.execute("""\n            SELECT *\n            FROM auditoria_cambios\n            WHERE tabla_afectada=%s\n              AND registro_id=%s\n            ORDER BY fecha_hora;\n        """,(tabla,registro_id))\n\n        return cur.fetchall()\n\n    finally:\n        cur.close()\n\n# La capa FHIR se incorporará después en un bloque separado.\n'

ruta=Path('main.py')
ruta.write_text(MAIN_CODE,encoding='utf-8')
print(f'main.py actualizado: {ruta.resolve()}')


main.py actualizado: C:\Users\yenpa\Desktop\Salud digital\proyecto_triaje\main.py


## Validar sintaxis


In [3]:
import py_compile
py_compile.compile('main.py',doraise=True)
print('Sintaxis de main.py correcta.')


Sintaxis de main.py correcta.


## Ejecutar la API

Después de generar y validar `main.py`, ejecútala desde PowerShell con:

```powershell
uvicorn main:app --reload
```

Swagger estará en `http://127.0.0.1:8000/docs`.
